## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import colorcet as cc
import yaml
from statsmodels.distributions.copula.api import GumbelCopula

import gumbel_copula_2dRP as rp
import RP_plotting as rp_plot
import regional_declustering as declust
# import gumbel_copula_ESS as ess


# set up plot preferences
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['font.size'] = 8

### Constants

In [ ]:
def load_config(config_path):
    '''Loads the configuration from a YAML file.'''
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

config = load_config('config/config.yaml')

In [ ]:
# Config parameters
# THRESH = config['THRESH']
THRESH=30
REGIONS = config['REGIONS']
MODEL = config['MODEL']
T = config['T']
n = config['n']

# Other parameters
EXPORTS = False
cmap = cc.cm['kbc_r']
u_vals = np.linspace(0.8, 0.999, n)
v_vals = np.linspace(0.8, 0.999, n)

# Sensitivity analysis
FRAC = [0.1, 0.2, 0.3, 0.4, 0.5]
BUFFER = [1, 2, 5, 7, 10, 14]

### Data

In [ ]:
# Drought summary data
if MODEL == 'obs':
    # Obs
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/Drought_Properties_jd.csv',
        parse_dates=['start', 'end', 'previous_end'],
        index_col='StaID'
        )
elif MODEL == 'nwm':
    # NWM
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/nwm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
else:
    # CBRFC
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/cbrfc_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )

# List of study gages with HCDN clusters
gages = pd.read_csv(
    '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/CRB Gages/NWM_v3_CRB_with_HCDN_cluster.csv',
    index_col='USGS_ID'
    )[['Lat', 'Lon', 'region']]

# Regions
regions = gages['region'].unique().tolist()
region_names = [
    'Southwest',
    'California and Interior West',
    'Rocky Mountains'
]
# print(regions[REGION])

## Sensitivity Analysis

### Number of regional events

In [ ]:
all_n = []

for REGION in REGIONS:
    
    temp = []
    
    # cycle through fraction
    for frac in FRAC:
        
        inter = []
        
        # cycle through buffer
        for buff in BUFFER:
    
            # Filter to region
            events = drought_data[drought_data['threshold'] == THRESH]
            gages_in_region = gages[gages['region'] == regions[REGION]].index.tolist()
            events = events[events.index.isin(gages_in_region)]
            events = events[['start', 'end', 'severity', 'duration']]
            data.dropna(inplace=True)
            
            # Compute the regional events
            regional_events = declust.regional_concurrence_intervals(
                events,
                frac_thresh = frac,
                buffer = buff
                )
            
            inter.append(len(regional_events))

        temp.append(inter)
    
    all_n.append(np.array(temp))

In [ ]:
all_n[2]

## Export the data

In [ ]:
# if EXPORTS:
#     df = {
#         'Region':       [x for x in region_names for _ in range(5)],
#         'T':            T * 3,
#         'Duration':     [item for sublist in out_duration for item in sublist],
#         'Severity':     [item for sublist in out_severity for item in sublist],
#         'D_upper':      [item for sublist in out_upper_d for item in sublist],
#         'D_lower':      [item for sublist in out_lower_d for item in sublist],
#         'S_upper':      [item for sublist in out_upper_s for item in sublist],
#         'S_lower':      [item for sublist in out_lower_s for item in sublist]
#     }

#     df = pd.DataFrame(df)
#     df = df.round(0)
    
#     df.to_csv(f'RP_Figures/New_style/RP_data_{MODEL}_{THRESH}.csv', index=False)